# ResNet-50 Multilabel UCMerced — Stratified vs Random Split

## Hipótesis
Los modelos ResNet-50 pre-entrenados en ImageNet son suficientemente robustos al desbalance de clases como para **no** degradarse significativamente cuando el split train/val es aleatorio en lugar de estratificado.
- **H0 (hipótesis verdadera):** Δ métricas entre ambos modelos < 2 pp → el pre-entrenamiento compensa el desbalance.
- **H1 (hipótesis falsa):** Δ métricas ≥ 2 pp → el split estratificado sigue siendo necesario.

Ambos modelos son ResNet-50 idénticos (misma seed, mismos pesos iniciales), solo difiere el tipo de split de train/val.

## 1. Setup — Drive, dependencias y repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())

\!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image wandb iterative-stratification torchmetrics

In [ ]:
# Run this everytime you update something in the repo\!
REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

if not os.listdir(PROJECT_DIR):
    \!git clone {REPO} {PROJECT_DIR}
else:
    \!git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("Working in:", os.getcwd())

## 2. Descarga del dataset UCMerced

In [ ]:
import zipfile, subprocess, shutil

if not os.path.exists('ucmdata'):
    print("Cloning ucmdata repo and extracting images...")
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')
    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zf:
        zf.extractall('UCMImages')
    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
    os.chdir(PROJECT_DIR)
    print("Dataset ready.")
else:
    print("Dataset already present.")

## 3. Importaciones

In [ ]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as tvm
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import utils
importlib.reload(utils)
from utils import (
    build_dataloaders,
    LightningModuleMultilabel,
    compute_test_metrics,
    append_metrics_to_csv,
    plot_training_curves,
    plot_per_class_metrics,
    plot_exact_match_by_class_count,
)

L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
Path("outputs/checkpoints").mkdir(parents=True, exist_ok=True)
Path("outputs/logs").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/predictions").mkdir(parents=True, exist_ok=True)
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Hiperparámetros

In [ ]:
PRETRAINED_MODEL      = "resnet50"
MAX_EPOCHS            = 25
EARLYSTOPPING_EPOCHS  = 5
LR                    = 1e-4
WEIGHT_DECAY          = 1e-5
THRESHOLD             = 0.5
SEED                  = 42
BATCH_SIZE            = 32
NUM_WORKERS           = 2

print(f"Model:        {PRETRAINED_MODEL}")
print(f"Max epochs:   {MAX_EPOCHS}  |  Early stopping: {EARLYSTOPPING_EPOCHS}")
print(f"LR:           {LR}  |  Weight decay: {WEIGHT_DECAY}")
print(f"Batch size:   {BATCH_SIZE}  |  Threshold: {THRESHOLD}")

## 5. Dataloaders — Test set compartido + Modelo A (Stratified) y Modelo B (Random)

Para garantizar comparabilidad, **ambos modelos se evalúan sobre el mismo test set**.
El procedimiento es:
1. Extraer el test set una sola vez con split estratificado → `shared_test_idx`
2. Modelo A: train/val estratificado + test compartido
3. Modelo B: train/val aleatorio + mismo test compartido

Solo difiere el split de train/val; el test set es idéntico.

In [ ]:
# ── Paso 1: Extraer test set compartido una sola vez (split estratificado) ──────
print("=" * 60)
print("  PASO 1: TEST SET COMPARTIDO (estratificado, seed fija)")
print("=" * 60)

_, _, test_loader, classes, _ = build_dataloaders(
    root_dir    = "ucmdata",
    label_file  = "LandUse_Multilabeled.txt",
    image_size  = (224, 224),
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    val_frac    = 0.15,
    test_frac   = 0.15,
    seed        = SEED,
    image_ext   = ".tif",
    stratified  = True,
)
shared_test_idx = np.array(test_loader.dataset.indices)
NUM_CLASSES = len(classes)
print(f"Shared test set : {len(shared_test_idx)} images  |  Classes: {NUM_CLASSES}")

# ── Paso 2A: Modelo A — train/val estratificado + test compartido ─────────────
print()
print("=" * 60)
print("  PASO 2A: DATALOADER A — STRATIFIED TRAIN/VAL")
print("=" * 60)

train_loader_A, val_loader_A, _, classes, pos_w_A = build_dataloaders(
    root_dir    = "ucmdata",
    label_file  = "LandUse_Multilabeled.txt",
    image_size  = (224, 224),
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    val_frac    = 0.15,
    test_frac   = 0.15,
    seed        = SEED,
    image_ext   = ".tif",
    stratified  = True,
    test_idx    = shared_test_idx,
)
print(f"Train: {len(train_loader_A.dataset)} | Val: {len(val_loader_A.dataset)} | Test (shared): {len(test_loader.dataset)}")

In [ ]:
# ── Paso 2B: Modelo B — train/val aleatorio + test compartido ────────────────
print("=" * 60)
print("  PASO 2B: DATALOADER B — RANDOM TRAIN/VAL")
print("=" * 60)

train_loader_B, val_loader_B, _, _, pos_w_B = build_dataloaders(
    root_dir    = "ucmdata",
    label_file  = "LandUse_Multilabeled.txt",
    image_size  = (224, 224),
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    val_frac    = 0.15,
    test_frac   = 0.15,
    seed        = SEED,
    image_ext   = ".tif",
    stratified  = False,
    test_idx    = shared_test_idx,
)
print(f"Train: {len(train_loader_B.dataset)} | Val: {len(val_loader_B.dataset)} | Test (shared): {len(test_loader.dataset)}")
print()
print("Test set es identico para A y B -> comparacion valida.")

## 6. Arquitectura ResNet-50

Se reemplaza la capa fully connected (`model.fc`) por una nueva `Linear(2048 → 17)`.
Todos los pesos permanecen entrenables (fine-tuning completo).

In [ ]:
def build_resnet50(num_classes: int):
    """ResNet-50 pre-trained on ImageNet with new multilabel classification head."""
    weights = tvm.ResNet50_Weights.IMAGENET1K_V2
    model   = tvm.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

_tmp = build_resnet50(NUM_CLASSES)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"ResNet-50  |  Trainable params: {n_params:,}  |  Output logits: {NUM_CLASSES}")
del _tmp

## 7. Entrenamiento — Modelo A (Stratified Split)

In [ ]:
print("=" * 55)
print("  MODELO A - STRATIFIED SPLIT")
print("=" * 55)

L.seed_everything(SEED, workers=True)

backbone_A = build_resnet50(NUM_CLASSES)
lit_A = LightningModuleMultilabel(
    model        = backbone_A,
    num_classes  = NUM_CLASSES,
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
    max_epochs   = MAX_EPOCHS,
    threshold    = THRESHOLD,
    pos_weight   = pos_w_A,
)

ckpt_A = ModelCheckpoint(
    dirpath  = "outputs/checkpoints",
    filename = "resnetA_stratified-best-{epoch:02d}-{val_f1:.4f}",
    monitor  = "val_f1", mode="max", save_top_k=1, save_weights_only=True,
)
early_A = EarlyStopping(
    monitor  = "val_f1", mode="max",
    patience = EARLYSTOPPING_EPOCHS, min_delta=1e-3, verbose=True,
)
logger_A = CSVLogger("outputs/logs", name="resnetA_stratified")

trainer_A = L.Trainer(
    max_epochs        = MAX_EPOCHS,
    accelerator       = "auto",
    devices           = "auto",
    callbacks         = [ckpt_A, early_A],
    logger            = logger_A,
    log_every_n_steps = 5,
)
trainer_A.fit(lit_A, train_loader_A, val_loader_A)

## 8. Entrenamiento — Modelo B (Random Split)

In [ ]:
print("=" * 55)
print("  MODELO B - RANDOM SPLIT")
print("=" * 55)

L.seed_everything(SEED, workers=True)   # misma seed -> misma inicializacion de pesos

backbone_B = build_resnet50(NUM_CLASSES)
lit_B = LightningModuleMultilabel(
    model        = backbone_B,
    num_classes  = NUM_CLASSES,
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
    max_epochs   = MAX_EPOCHS,
    threshold    = THRESHOLD,
    pos_weight   = pos_w_B,
)

ckpt_B = ModelCheckpoint(
    dirpath  = "outputs/checkpoints",
    filename = "resnetB_random-best-{epoch:02d}-{val_f1:.4f}",
    monitor  = "val_f1", mode="max", save_top_k=1, save_weights_only=True,
)
early_B = EarlyStopping(
    monitor  = "val_f1", mode="max",
    patience = EARLYSTOPPING_EPOCHS, min_delta=1e-3, verbose=True,
)
logger_B = CSVLogger("outputs/logs", name="resnetB_random")

trainer_B = L.Trainer(
    max_epochs        = MAX_EPOCHS,
    accelerator       = "auto",
    devices           = "auto",
    callbacks         = [ckpt_B, early_B],
    logger            = logger_B,
    log_every_n_steps = 5,
)
trainer_B.fit(lit_B, train_loader_B, val_loader_B)

## 9. Evaluacion en test — Modelo A (Stratified)

In [ ]:
print("=" * 55)
print("  EVALUACION MODELO A - STRATIFIED SPLIT")
print("=" * 55)

trainer_A.test(lit_A, dataloaders=test_loader, ckpt_path="best")

best_path_A = ckpt_A.best_model_path
print(f"Best checkpoint: {best_path_A}")
lit_A = LightningModuleMultilabel.load_from_checkpoint(best_path_A, model=lit_A.model)

preds_out_A = trainer_A.predict(lit_A, dataloaders=test_loader)
probs_A  = torch.cat([b["probs"]  for b in preds_out_A]).numpy()
preds_A  = torch.cat([b["preds"]  for b in preds_out_A]).numpy()
labels_A = torch.cat([b["labels"] for b in preds_out_A]).numpy()

metrics_A = compute_test_metrics(preds_A, labels_A, probs_A)
print("\nModel A (Stratified) — Test Metrics:")
for k, v in metrics_A.items():
    print(f"  {k:<15}: {v:.4f}")

append_metrics_to_csv(metrics_A, model_name="ResNet50_Stratified")

## 10. Evaluacion en test — Modelo B (Random)

In [ ]:
print("=" * 55)
print("  EVALUACION MODELO B - RANDOM SPLIT")
print("=" * 55)

trainer_B.test(lit_B, dataloaders=test_loader, ckpt_path="best")

best_path_B = ckpt_B.best_model_path
print(f"Best checkpoint: {best_path_B}")
lit_B = LightningModuleMultilabel.load_from_checkpoint(best_path_B, model=lit_B.model)

preds_out_B = trainer_B.predict(lit_B, dataloaders=test_loader)
probs_B  = torch.cat([b["probs"]  for b in preds_out_B]).numpy()
preds_B  = torch.cat([b["preds"]  for b in preds_out_B]).numpy()
labels_B = torch.cat([b["labels"] for b in preds_out_B]).numpy()

metrics_B = compute_test_metrics(preds_B, labels_B, probs_B)
print("\nModel B (Random) — Test Metrics:")
for k, v in metrics_B.items():
    print(f"  {k:<15}: {v:.4f}")

append_metrics_to_csv(metrics_B, model_name="ResNet50_Random")

## 11. Tabla comparativa y veredicto de hipotesis

In [ ]:
comparison_df = pd.DataFrame({
    "Metric":     list(metrics_A.keys()),
    "Stratified": [round(v, 4) for v in metrics_A.values()],
    "Random":     [round(v, 4) for v in metrics_B.values()],
})
comparison_df["Delta (Random - Strat)"] = (
    comparison_df["Random"] - comparison_df["Stratified"]
).round(4)

print("\n" + "=" * 55)
print("  STRATIFIED vs RANDOM - COMPARATIVA FINAL")
print("=" * 55)
print(comparison_df.to_string(index=False))

delta_f1  = abs(metrics_A["macro_f1"] - metrics_B["macro_f1"])
delta_map = abs(metrics_A["macro_map"] - metrics_B["macro_map"])

print(f"\nDelta macro-F1  = {delta_f1:.4f}")
print(f"Delta macro-mAP = {delta_map:.4f}")

if delta_f1 < 0.02:
    print("\nH0 no rechazada: Delta < 2 pp -> el pre-entrenamiento compensa el desbalance.")
else:
    print("\nH1 confirmada: Delta >= 2 pp -> el split estratificado sigue siendo necesario.")

## 12. Curvas de aprendizaje

In [ ]:
plot_training_curves(
    logger_A,
    model_name = "ResNet-50 - Stratified Split",
    save_path  = "outputs/figures/resnetA_stratified_curves.png",
)

plot_training_curves(
    logger_B,
    model_name = "ResNet-50 - Random Split",
    save_path  = "outputs/figures/resnetB_random_curves.png",
)

## 13. Metricas per-class — F1 y AP por clase

In [ ]:
plot_per_class_metrics(
    labels_A, preds_A, probs_A, classes,
    macro_f1   = metrics_A["macro_f1"],
    macro_map  = metrics_A["macro_map"],
    model_name = "ResNet-50 Stratified",
    save_path  = "outputs/figures/resnetA_per_class.png",
    csv_output = "outputs/resnetA_per_class.csv",
)

plot_per_class_metrics(
    labels_B, preds_B, probs_B, classes,
    macro_f1   = metrics_B["macro_f1"],
    macro_map  = metrics_B["macro_map"],
    model_name = "ResNet-50 Random",
    save_path  = "outputs/figures/resnetB_per_class.png",
    csv_output = "outputs/resnetB_per_class.csv",
)

## 14. Guardar predicciones para analisis posterior

Guarda `preds`, `labels` y `probs` de ambos modelos en `outputs/predictions/` como archivos `.npz`.
De esta forma se pueden cargar en un script aparte sin necesidad de re-entrenar.

In [ ]:
PRED_DIR = Path(PROJECT_DIR) / "outputs" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Modelo A - Stratified
np.savez(
    str(PRED_DIR / "resnetA_stratified.npz"),
    preds   = preds_A,
    labels  = labels_A,
    probs   = probs_A,
    classes = np.array(classes),
)
print(f"Saved: {PRED_DIR / 'resnetA_stratified.npz'}")

# Modelo B - Random
np.savez(
    str(PRED_DIR / "resnetB_random.npz"),
    preds   = preds_B,
    labels  = labels_B,
    probs   = probs_B,
    classes = np.array(classes),
)
print(f"Saved: {PRED_DIR / 'resnetB_random.npz'}")

### Como cargar y graficar en un script aparte

In [ ]:
# Cargar datos guardados
PRED_DIR = Path(PROJECT_DIR) / "outputs" / "predictions"

data_A = np.load(str(PRED_DIR / "resnetA_stratified.npz"), allow_pickle=True)
data_B = np.load(str(PRED_DIR / "resnetB_random.npz"),     allow_pickle=True)

preds_A_saved  = data_A["preds"];   labels_A_saved = data_A["labels"]
preds_B_saved  = data_B["preds"];   labels_B_saved = data_B["labels"]
classes_saved  = list(data_A["classes"])

print(f"Loaded A — preds: {preds_A_saved.shape}, labels: {labels_A_saved.shape}")
print(f"Loaded B — preds: {preds_B_saved.shape}, labels: {labels_B_saved.shape}")

# ── Plot individual Modelo A ──────────────────────────────────────────────────
fig_A, _ = plot_exact_match_by_class_count(
    preds_A_saved, labels_A_saved,
    model_name = "ResNet-50 Stratified",
    color      = "steelblue",
    save_path  = str(Path(PROJECT_DIR) / "outputs/figures/resnetA_em_by_classcount.png"),
)
display(fig_A)
plt.close(fig_A)

# ── Plot individual Modelo B ──────────────────────────────────────────────────
fig_B, _ = plot_exact_match_by_class_count(
    preds_B_saved, labels_B_saved,
    model_name = "ResNet-50 Random",
    color      = "coral",
    save_path  = str(Path(PROJECT_DIR) / "outputs/figures/resnetB_em_by_classcount.png"),
)
display(fig_B)
plt.close(fig_B)

# ── Plot comparativo side-by-side ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

plot_exact_match_by_class_count(
    preds_A_saved, labels_A_saved,
    model_name = "ResNet-50 Stratified",
    color      = "steelblue",
    ax         = axes[0],
)
plot_exact_match_by_class_count(
    preds_B_saved, labels_B_saved,
    model_name = "ResNet-50 Random",
    color      = "coral",
    ax         = axes[1],
)

fig.suptitle("Exact Match Accuracy by Number of Classes per Image — Stratified vs Random",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(
    str(Path(PROJECT_DIR) / "outputs/figures/resnet_em_by_classcount_comparison.png"),
    dpi=150, bbox_inches="tight"
)
display(fig)
plt.close(fig)

## 15. Referencias

- He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep residual learning for image recognition*. IEEE CVPR 2016.
- Yang, Y., & Newsam, S. (2010). *Bag-of-visual-words and spatial extensions for land-use classification*. ACM SIGSPATIAL.
- UCMerced Land Use Dataset: http://weegee.vision.ucmerced.edu/datasets/landuse.html